# Clean XGBoost + LightGBM ensemble

A self-contained, honest blend of the two tree models. To avoid the mismatch from the
registry (XGBoost was refit on all data, LightGBM used a different `split_date`), we:

1. Load the data once and use **the same** split (`2012-01-01`) for both models.
2. Train each model **fresh on train-only**, with its own cloned copy of the shared
   preprocessor (so no fitted-state or feature mismatch).
3. Predict both on **the same** validation rows.
4. Grid-search the blend weight to minimise WMAE.

This is the weight-selection step; for a Kaggle submission you would refit the chosen
blend on all data and apply it to `test.csv`.

In [1]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print('cwd:', os.getcwd())

cwd: D:\Desktop\ML Project


In [2]:
import warnings
import numpy as np
import pandas as pd
from sklearn.base import clone

from src.features.preprocessing import get_model_ready_data
from src.models.xgboost_pipeline import create_xgboost_pipeline
from src.models.lightgbm_pipeline import create_lightgbm_pipeline
from src.evaluation.metrics import calculate_wmae

warnings.filterwarnings('ignore')

SPLIT_DATE = '2012-01-01'  # same split for BOTH models -> aligned, honest validation
train_raw = pd.read_csv('data/train.csv')
stores = pd.read_csv('data/stores.csv')
features = pd.read_csv('data/features.csv')

X_train, y_train, X_val, y_val, is_holiday_val, preprocessor = get_model_ready_data(
    train_raw, stores, features, split_date=SPLIT_DATE
)
print('train:', X_train.shape, '| val:', X_val.shape)

train: (294132, 17) | val: (127438, 17)


## XGBoost (best config from the search), trained on train-only

In [3]:
xgb_params = {
    'objective': 'reg:squarederror',
    'n_estimators': 300, 'max_depth': 12, 'learning_rate': 0.05,
    'subsample': 0.8, 'colsample_bytree': 0.9,
    'random_state': 42, 'n_jobs': -1,
}
xgb_pipe = create_xgboost_pipeline(clone(preprocessor), xgb_params)
xgb_pipe.fit(X_train, y_train)
xgb_val = np.clip(xgb_pipe.predict(X_val), 0, None)
xgb_wmae = calculate_wmae(y_val, xgb_val, is_holiday_val)
print(f'XGBoost val WMAE: {xgb_wmae:,.2f}')

XGBoost val WMAE: 1,985.45


## LightGBM search (train-only)

A few stronger configs (deeper leaves, more estimators) to push LightGBM below its
single-fit baseline. Several minutes on CPU.

In [4]:
lgb_common = {'random_state': 42, 'n_jobs': -1, 'verbose': -1, 'subsample_freq': 1}
lgb_space = [
    {'n_estimators': 1000, 'learning_rate': 0.03, 'num_leaves': 255, 'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.9, 'max_depth': -1},
    {'n_estimators': 2000, 'learning_rate': 0.02, 'num_leaves': 511, 'min_child_samples': 5, 'subsample': 1.0, 'colsample_bytree': 0.9, 'max_depth': -1},
    {'n_estimators': 800, 'learning_rate': 0.05, 'num_leaves': 127, 'min_child_samples': 20, 'subsample': 0.8, 'colsample_bytree': 0.8, 'max_depth': -1},
    {'n_estimators': 1500, 'learning_rate': 0.03, 'num_leaves': 255, 'min_child_samples': 10, 'subsample': 0.9, 'colsample_bytree': 0.7, 'max_depth': 12},
]

best_lgb_wmae, best_lgb_val, best_lgb_params = float('inf'), None, None
for i, cand in enumerate(lgb_space, 1):
    params = {**lgb_common, **cand}
    pipe = create_lightgbm_pipeline(clone(preprocessor), params)
    pipe.fit(X_train, y_train)
    val_pred = np.clip(pipe.predict(X_val), 0, None)
    wmae = calculate_wmae(y_val, val_pred, is_holiday_val)
    print(f"LGB cfg{i} (leaves{cand['num_leaves']}, est{cand['n_estimators']}): WMAE {wmae:,.2f}")
    if wmae < best_lgb_wmae:
        best_lgb_wmae, best_lgb_val, best_lgb_params = wmae, val_pred, params
print('---')
print(f'Best LightGBM val WMAE: {best_lgb_wmae:,.2f}')

LGB cfg1 (leaves255, est1000): WMAE 2,166.44
LGB cfg2 (leaves511, est2000): WMAE 1,971.13
LGB cfg3 (leaves127, est800): WMAE 2,390.00
LGB cfg4 (leaves255, est1500): WMAE 2,172.54
---
Best LightGBM val WMAE: 1,971.13


## Blend weight search

In [5]:
best_w, best_blend_wmae = 0.0, float('inf')
for w in np.linspace(0, 1, 21):
    blend = w * xgb_val + (1 - w) * best_lgb_val
    wmae = calculate_wmae(y_val, blend, is_holiday_val)
    if wmae < best_blend_wmae:
        best_blend_wmae, best_w = wmae, w

print('=== Clean tree ensemble (same split, train-only fit) ===')
print(f'XGBoost solo : {xgb_wmae:,.2f}')
print(f'LightGBM solo: {best_lgb_wmae:,.2f}')
print(f'Best blend   : {best_blend_wmae:,.2f}  at w_xgb={best_w:.2f} (w_lgb={1 - best_w:.2f})')
print('TimesFM reference champion: 1,635')

=== Clean tree ensemble (same split, train-only fit) ===
XGBoost solo : 1,985.45
LightGBM solo: 1,971.13
Best blend   : 1,932.17  at w_xgb=0.45 (w_lgb=0.55)
TimesFM reference champion: 1,635


## Log the ensemble result to the shared DagsHub

In [6]:
import mlflow
import dagshub

dagshub.init(repo_owner='slosa23', repo_name='Walmart-Sales-Forecasting', mlflow=True)
mlflow.set_experiment('Tree_Ensemble_Blend')

with mlflow.start_run(run_name='XGB_LGB_Blend'):
    mlflow.log_param('split_date', SPLIT_DATE)
    mlflow.log_param('w_xgb', round(float(best_w), 3))
    mlflow.log_param('w_lgb', round(float(1 - best_w), 3))
    mlflow.log_params({f'xgb_{k}': v for k, v in xgb_params.items()})
    mlflow.log_params({f'lgb_{k}': v for k, v in best_lgb_params.items()})
    mlflow.log_metric('xgb_WMAE', float(xgb_wmae))
    mlflow.log_metric('lgb_WMAE', float(best_lgb_wmae))
    mlflow.log_metric('ensemble_WMAE', float(best_blend_wmae))
    print('Logged to DagsHub experiment Tree_Ensemble_Blend')

Accessing as GigiSichinava

Initialized MLflow to track repo "slosa23/Walmart-Sales-Forecasting"

Repository slosa23/Walmart-Sales-Forecasting initialized!

2026/07/12 21:57:10 INFO mlflow.tracking.fluent: Experiment with name 'Tree_Ensemble_Blend' does not exist. Creating a new experiment.


Logged to DagsHub experiment Tree_Ensemble_Blend
🏃 View run XGB_LGB_Blend at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/14/runs/f938d2929eac49cda86ba9335983f108
🧪 View experiment at: https://dagshub.com/slosa23/Walmart-Sales-Forecasting.mlflow/#/experiments/14


## Notes

- Both models trained on the **same** `2012-01-01` split, train-only, with cloned copies
  of the shared preprocessor -> no registry / preprocessor mismatch.
- WMAE via the shared `calculate_wmae`, comparable to every other model.
- Weights are chosen on validation. For a Kaggle submission, refit XGBoost and the best
  LightGBM on **all** data and apply `w_xgb`/`w_lgb` to the test predictions.
- TimesFM (1,635) is still the overall champion; this blend is the best tree-only model.